# **TALASH - Talent Acquisition & Learning Automation for Smart Hiring**
## **Course:** CS417 - Large Language Models
### **Milestone 1**


In [1]:
# Install Required Libraries
!pip install pypdf pandas openpyxl -q
print("Libraries installed successfully")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 10.3 MB/s eta 0:00:00
Libraries installed successfully



## **System Architecture**

The TALASH system has 6 layers:

```
LAYER 1 - INPUT        : Folder-based CV Ingestion (PDF files)
LAYER 2 - PREPROCESS   : PDF Text Extraction, Cleaning, Segmentation
LAYER 3 - EXTRACTION   : Personal, Education, Experience, Publications,
                         Awards, Patents, References (Regex/NLP)
LAYER 4 - LLM ANALYSIS : Prompt Building -> LLM API -> Scored Output
                         (Milestone 2)
LAYER 5 - STORAGE      : Excel (multi-sheet) + JSON + CSV
LAYER 6 - UI/OUTPUT    : Streamlit Web App (Milestone 2)
                         Jupyter Notebook (current)
```

**Design Decisions:**
- Regex-based extraction for Milestone 1 (no API needed, runs offline)
- LLM integration planned for Milestone 2 (GPT / LLaMA)
- Modular design: each section has its own extraction function
- Storage: Excel with 8 sheets + JSON for LLM use + CSV for quick view


In [2]:
# Imports and Configuration

import os
import re
import json
import pandas as pd
from pypdf import PdfReader
from datetime import datetime

# Configuration
CV_FOLDER    = "cvs"
OUTPUT_EXCEL = "talash_output.xlsx"
OUTPUT_JSON  = "talash_output.json"
OUTPUT_CSV   = "talash_personal.csv"

os.makedirs(CV_FOLDER, exist_ok=True)

print("=" * 50)
print("  TALASH - Milestone 1")
print("=" * 50)
print(f"CV Folder  : {CV_FOLDER}/")
print(f"Excel Out  : {OUTPUT_EXCEL}")
print(f"JSON Out   : {OUTPUT_JSON}")
print(f"CSV Out    : {OUTPUT_CSV}")
print(f"Run Time   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Configuration loaded OK")


  TALASH - Milestone 1
CV Folder  : cvs/
Excel Out  : talash_output.xlsx
JSON Out   : talash_output.json
CSV Out    : talash_personal.csv
Run Time   : 2026-04-13 15:18:56
Configuration loaded OK



## **Preprocessing Module**

This module handles CV ingestion:
1. Reads all PDF files from the `cvs/` folder
2. Extracts raw text from each page using `pypdf`
3. Cleans the text (removes control characters, extra spaces)
4. Splits multi-candidate PDFs into individual candidate blocks


In [3]:
# PDF Text Extraction and Preprocessing

def read_pdf_pages(path):
    """
    Reads a PDF file and returns a list of strings, one per page.
    """
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        pages.append(text.strip() if text else "")
    return pages


def clean_text(text):
    """
    Basic text cleaning:
    - Removes special control characters
    - Reduces extra whitespace
    """
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', ' ', text)
    text = re.sub(r' {3,}', '  ', text)
    return text.strip()


def is_candidate_start(text):
    """
    Checks if this page starts a new candidate record.
    Looks for 'Name' keyword at the start.
    """
    return bool(re.search(
        r'\bName\s+(?!of\b|Degree\b|Post\b|Contact\b|Qualification\b)([A-Za-z][\w\.\s]+)',
        text[:400]
    ))


def group_candidates(pages):
    """
    Groups pages that belong to the same candidate.
    Returns a list of text blocks (one block per candidate).
    """
    candidates, current = [], []
    for page in pages:
        if page and is_candidate_start(page):
            if current:
                candidates.append("\n".join(current))
            current = [page]
        elif page:
            current.append(page)
    if current:
        candidates.append("\n".join(current))
    return candidates


def preview_pdf(path, max_chars=800):
    """
    Preview first portion of a PDF - useful to verify extraction works.
    """
    pages = read_pdf_pages(path)
    full_text = "\n".join(pages)
    print(f"File   : {os.path.basename(path)}")
    print(f"Pages  : {len(pages)}")
    print(f"Chars  : {len(full_text)}")
    print("-" * 50)
    print("TEXT PREVIEW (first 800 chars):")
    print("-" * 50)
    print(full_text[:max_chars])
    print("...")
    return pages


# Try extracting from available PDFs
pdfs = [os.path.join(CV_FOLDER, f) for f in os.listdir(CV_FOLDER) if f.lower().endswith('.pdf')]
if pdfs:
    print(f"Found {len(pdfs)} PDF(s) in {CV_FOLDER}/\n")
    pages = preview_pdf(pdfs[0])
else:
    print(f"No PDFs found in {CV_FOLDER}/ folder.")
    print("Please add a CV PDF to the cvs/ folder and run this cell again.")


Found 1 PDF(s) in cvs/

File   : Handler (8).pdf
Pages  : 251
Chars  : 210779
--------------------------------------------------
TEXT PREVIEW (first 800 chars):
--------------------------------------------------
Professional Qualification
Name MUHAMMAD SALMAN
QAMAR
Father's /GuardianQAMAR UZ ZAMAN
Date/Place of
Birth:
10-Nov-1988 /
PAKISTANI
Father’s
Occupation:
Govt Servant
Spouse Name:Aamna Bibi Spouse
Occupation:
Other
Current Salary:200000 Marital Status:Married
Unit / Corps:: Course:
SOD: SOS:
Expected Salary:200000 Present
Employment:
Associate Professor at Xi'an International University, China ,
Shaanxi, China since 01/01/2025
Serving/Retired
Education
Name of Degree Specialization Grade / %age / GPA Passing Year Board/University
PhD in Electrical EngineeringCommunication Systems3.33 2024 International Islamic university, Islamabad
MS in Electrical EngineeringCommunication Systems3.03 2014 COMSATS University, Islamabad, Attock Campus
BSc Electronics EngineeringElectronics 69.84 


## **Section Extraction Functions (NLP Pipeline)**

Each function extracts one specific section from a candidate's text.
This is the **NLP pipeline** using Regex pattern matching.

> **Note:** In Milestone 2, these will be enhanced or replaced by LLM-based
> extraction for better accuracy on varied CV formats.


In [4]:
# Section Extraction Functions

def get_field(pattern, text, default='N/A'):
    """Helper: extract a field using regex."""
    m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    return m.group(1).strip() if m else default


# Personal Information
def extract_personal(text, cid, source):
    """Extracts name, DOB, employment, salary, etc."""
    name = 'N/A'
    m = re.search(
        r"\bName\s+(?!of\b|Degree\b|Post\b|Contact\b|Qualification\b)(.+?)(?:Father's|/Guardian)",
        text, re.IGNORECASE | re.DOTALL
    )
    if m:
        raw = m.group(1)
        parts = []
        for line in raw.split('\n'):
            line = line.strip()
            if not line: continue
            if re.search(r'(Father|Guardian|Date|Spouse|Salary|Marital|Unit|Corps)', line, re.IGNORECASE): break
            parts.append(line)
        name = ' '.join(parts).strip()
    name = re.sub(r'\s*(Father|Guardian|Date|Spouse|Salary|Unit).*', '', name, flags=re.IGNORECASE).strip()

    dob_m = re.search(r'Date/Place of\s*(?:\n\s*)?Birth:\s*([\d\-A-Za-z/ ]+)', text)
    dob   = dob_m.group(1).strip() if dob_m else 'N/A'

    emp_m = re.search(r'Present\s*\nEmployment:\s*\n?(.+?)\nServing', text, re.DOTALL)
    if not emp_m:
        emp_m = re.search(r'Present Employment:\s*(.+?)\nServing', text, re.DOTALL)
    employment = emp_m.group(1).replace('\n', ' ').strip() if emp_m else 'N/A'

    return {
        'candidate_id':       cid,
        'Name':               name if name else 'N/A',
        'Date of Birth':      dob,
        'Marital Status':     get_field(r'Marital Status[:\s]+(\w[\w\s\-]+?)(?:\n|Unit)', text),
        'Current Salary':     get_field(r'Current Salary[:\s]+([\w,\.]+)', text),
        'Expected Salary':    get_field(r'Expected Salary[:\s]+([\w,\.]+)', text),
        'Present Employment': employment,
        'Applying For':       get_field(r'Candidate for the Post of\s+(.+?)(?:\(Apply|\n)', text),
        'Apply Date':         get_field(r'Apply Date:\s*([\d\-A-Za-z]+)', text),
        'Source File':        source,
    }


# Education
def extract_education(text, cid):
    """Extracts education rows: degree, year, CGPA."""
    rows = []
    edu_m = re.search(
        r'Education\s*\nName of Degree\s+Specialization.+?\n(.*?)(?:Candidate for the Post|Civil Experience|Professional Qualification\nCivil)',
        text, re.DOTALL | re.IGNORECASE
    )
    if not edu_m: return rows
    for line in edu_m.group(1).split('\n'):
        line = line.strip()
        if not line or len(line) < 5: continue
        if re.match(r'(Name of|Board/Uni|Passing|Speciali|Grade)', line, re.IGNORECASE): continue
        year_m  = re.search(r'\b(19|20)(\d{2})\b', line)
        grade_m = re.search(r'\b(\d+\.?\d*)\s+(?=\d{4})', line)
        if not grade_m: grade_m = re.search(r'\b(\d{2,3}\.?\d*)\b', line)
        if year_m:
            rows.append({'candidate_id': cid, 'Degree/Details': line[:100],
                         'Year': year_m.group(0), 'Grade/CGPA': grade_m.group(1) if grade_m else 'N/A'})
    return rows


# Professional Qualifications
def extract_prof_qual(text, cid):
    """Extracts certifications and professional qualifications."""
    rows = []
    pq_m = re.search(
        r'(?:Professional Qualification\s*\n)?Name of Qualification[:\s]*Passing Year[:\s]*Institution Name[:\s]*\n(.*?)(?:Name of Post|Civil Experience\s*\nName|$)',
        text, re.DOTALL | re.IGNORECASE
    )
    if not pq_m: return rows
    for line in pq_m.group(1).split('\n'):
        line = line.strip()
        if not line or len(line) < 5: continue
        year_m = re.search(r'\b(19|20)\d{2}\b', line)
        rows.append({'candidate_id': cid, 'Qualification': line[:150], 'Year': year_m.group(0) if year_m else 'N/A'})
    return rows


# Work Experience
def extract_experience(text, cid):
    """Extracts job history: position, organization, duration."""
    rows = []
    exp_m = re.search(
        r'Name of Post\s+Organization\s+Location\s+Duration of Employment\s*\n(.*?)(?:Paper Title|Research\nPublications|Awards|References|Patents|$)',
        text, re.DOTALL | re.IGNORECASE
    )
    if not exp_m: return rows
    date_pat = r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)-(\d{4})'
    for line in exp_m.group(1).split('\n'):
        line = line.strip()
        if not line or len(line) < 5: continue
        dates = re.findall(date_pat, line, re.IGNORECASE)
        has_present = bool(re.search(r'\bPresent\b', line, re.IGNORECASE))
        if not dates and not has_present: continue
        date_idx = re.search(date_pat, line, re.IGNORECASE)
        role_org  = line[:date_idx.start()].strip() if date_idx else line
        date_part = line[date_idx.start():].strip() if date_idx else 'Present'
        rows.append({'candidate_id': cid, 'Position & Org': role_org[:100], 'Duration': date_part[:50]})
    return rows


# Research Publications
def extract_publications(text, cid):
    """Extracts publications: title, journal, impact factor, date."""
    rows = []
    pub_m = re.search(
        r'Paper Title\s+Name of Author.+?\n(.*?)(?:Name\s+Contact\s+Type|References?\s*\n\s*Name\s+Contact|$)',
        text, re.DOTALL | re.IGNORECASE
    )
    if not pub_m: return rows
    lines = pub_m.group(1).split('\n')
    year_pat    = re.compile(r'^(20\d{2})$')
    month_pat   = re.compile(r'^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)-?$', re.IGNORECASE)
    journal_pat = re.compile(r'^(International Journal|International Conference|Journal|Conference|IEEE|Springer|MDPI|Elsevier|ACM|Wiley|Hindawi|Sensors|Electronics|Energies)', re.IGNORECASE)
    impact_pat  = re.compile(r'\b(\d+\.\d+)\b')
    groups, current_group = [], []
    for line in lines:
        line = line.strip()
        if not line: continue
        if re.match(r'(Name of|Paper Title|Factor|Vol\s|PP\s|^Vol$|^PP$|^Date$)', line, re.IGNORECASE): continue
        current_group.append(line)
        if year_pat.match(line):
            groups.append(current_group[:])
            current_group = []
    if current_group: groups.append(current_group)
    for group in groups:
        if not group: continue
        journal_idx = -1
        for idx, line in enumerate(group):
            if impact_pat.search(line) or journal_pat.match(line):
                journal_idx = idx; break
        if journal_idx < 0: continue
        title        = ' '.join(group[:journal_idx]).strip()
        journal_line = group[journal_idx]
        month, year  = '', ''
        for l in group[journal_idx+1:]:
            if month_pat.match(l): month = l.replace('-','').strip()
            if year_pat.match(l):  year  = l.strip()
        inline = re.search(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)-?(\d{4})', journal_line, re.IGNORECASE)
        if inline: month, year = inline.group(1), inline.group(2)
        date_str = f'{month}-{year}' if month and year else (year if year else 'N/A')
        im = impact_pat.search(journal_line)
        rows.append({'candidate_id': cid, 'Title': title[:200], 'Published In': journal_line[:100],
                     'Impact Factor': im.group(1) if im else 'N/A', 'Date': date_str})
    return rows


# Awards & Scholarships
def extract_awards(text, cid):
    """Extracts awards and scholarships."""
    rows = []
    aw_m = re.search(r'Awards\s*[&and]*\s*Scholarships?\s*\n(?:Type\s+Detail\s*\n)?(.*?)(?:References?\s*\nName|Patents?\s*\n|$)', text, re.DOTALL | re.IGNORECASE)
    if not aw_m: return rows
    entries = re.split(r'\n(?=(?:Merit Scholarships?|Academic Awards?|Membership)\s)', aw_m.group(1))
    for entry in entries:
        entry = entry.strip()
        if len(entry) < 5: continue
        type_m = re.match(r'(Merit Scholarships?|Academic Awards?|Membership)\s+(.*)', entry, re.DOTALL)
        rows.append({'candidate_id': cid,
                     'Type':   type_m.group(1).strip() if type_m else 'Other',
                     'Detail': (type_m.group(2) if type_m else entry).replace('\n',' ').strip()[:300]})
    return rows


# Patents
def extract_patents(text, cid):
    """Extracts patent details."""
    rows = []
    pat_m = re.search(r'\bPatents?\b\s*\n(?:No\.?\s+Patent Title\s*\n)?(.*?)(?:\bReferences?\b\s*\nName|\bAwards\b|$)', text, re.DOTALL | re.IGNORECASE)
    if not pat_m: return rows
    pat_text = pat_m.group(1).strip()
    if re.match(r'^(NA|N/A|0\s+NA|No\s+NA)\s*$', pat_text, re.IGNORECASE): return rows
    full_text = pat_text.replace('\n', ' ')
    if re.search(r'(Impact Factor|Vol PP Date|Paper Title)', full_text, re.IGNORECASE): return rows
    for entry in re.split(r'(?=GE-\d+|\b\d+\.\s+[A-Z])', full_text):
        entry = entry.strip()
        if len(entry) > 10 and not re.match(r'^(NA|N/A|0|No)\s*$', entry, re.IGNORECASE):
            rows.append({'candidate_id': cid, 'Patent Details': entry[:300]})
    return rows


# References
def extract_references(text, cid):
    """Extracts reference contacts."""
    rows = []
    ref_m = re.search(r'Name\s+Contact\s*\nType\s+Designation\s+Address.+?\n(.*?)$', text, re.DOTALL | re.IGNORECASE)
    if not ref_m: ref_m = re.search(r'Name\s+Contact\s+Type\s+Designation\s+Address.+?\n(.*?)$', text, re.DOTALL | re.IGNORECASE)
    if not ref_m: return rows
    entries = re.findall(r'([A-Z][\w\.\s,]+?)\s*Reference\s+(.+?)(?=(?:[A-Z][\w\.\s,]+?\s*Reference)|$)', ref_m.group(1), re.DOTALL)
    for name, details in entries:
        d = details.replace('\n', ' ').strip()
        email_m = re.search(r'[\w\.\-]+@[\w\.\-]+\.\w+', d)
        phone_m = re.search(r'(\+?\d[\d\s\-]{6,})', d)
        rows.append({'candidate_id': cid, 'Name': name.strip()[:80], 'Details': d[:200],
                     'Email': email_m.group(0) if email_m else 'N/A',
                     'Phone': phone_m.group(1).strip() if phone_m else 'N/A'})
    return rows

print("All extraction functions loaded OK")


All extraction functions loaded OK



## Main Processing Pipeline

This cell runs the full pipeline:
1. Finds all PDFs in `cvs/` folder
2. Extracts and segments candidates
3. Runs all section extraction functions
4. Returns a combined dictionary of all results


In [5]:
# Main Processing Pipeline

def process_one_pdf(pdf_path):
    """Process one PDF and return all extracted sections."""
    print(f'  Reading: {os.path.basename(pdf_path)}')
    pages      = read_pdf_pages(pdf_path)
    candidates = group_candidates(pages)
    print(f'  Candidates found: {len(candidates)}')

    result = {k: [] for k in ['personal','education','prof_qual','experience','publications','awards','patents','references']}

    for i, block in enumerate(candidates, 1):
        src = os.path.basename(pdf_path)
        result['personal'].append(extract_personal(block, i, src))
        result['education'].extend(extract_education(block, i))
        result['prof_qual'].extend(extract_prof_qual(block, i))
        result['experience'].extend(extract_experience(block, i))
        result['publications'].extend(extract_publications(block, i))
        result['awards'].extend(extract_awards(block, i))
        result['patents'].extend(extract_patents(block, i))
        result['references'].extend(extract_references(block, i))
        name = result['personal'][-1]['Name']
        print(f'    [{i:02d}] {name[:50]}')

    return result


def run_pipeline():
    """Full TALASH preprocessing pipeline."""
    print('=' * 55)
    print('  TALASH - Running Preprocessing Pipeline')
    print('=' * 55)

    pdfs = [os.path.join(CV_FOLDER, f) for f in sorted(os.listdir(CV_FOLDER)) if f.lower().endswith('.pdf')]

    if not pdfs:
        print(f'No PDFs found in {CV_FOLDER}/')
        print('Please add CV PDFs to the cvs/ folder.')
        return None

    print(f'[Step 1] Found {len(pdfs)} PDF(s):')
    for p in pdfs: print(f'         - {os.path.basename(p)}')

    print('\n[Step 2] Extracting data...')
    combined = {k: [] for k in ['personal','education','prof_qual','experience','publications','awards','patents','references']}
    for pdf in pdfs:
        r = process_one_pdf(pdf)
        for k in combined: combined[k].extend(r[k])

    print('\n[Step 3] Summary:')
    labels = {'personal':'Candidates','education':'Education rows','prof_qual':'Prof Qual rows',
              'experience':'Experience rows','publications':'Publication rows',
              'awards':'Award rows','patents':'Patent rows','references':'Reference rows'}
    for k, label in labels.items():
        icon = 'OK' if len(combined[k]) > 0 else '--'
        print(f'    [{icon}] {label:<22}: {len(combined[k])}')

    return combined


# Run it
extracted_data = run_pipeline()


  TALASH - Running Preprocessing Pipeline
[Step 1] Found 1 PDF(s):
         - Handler (8).pdf

[Step 2] Extracting data...
  Reading: Handler (8).pdf
  Candidates found: 43
    [01] MUHAMMAD SALMAN QAMAR
    [02] Aamina Akbar
    [03] MUHAMMAD SHAHWAZ
    [04] shaheer
    [05] Nasir Ali Shah
    [06] Muhammad Farrukh Qureshi
    [07] Idris Khan
    [08] MUHAMMAD MAJID AZIZ
    [09] Muhammad Sarmad Tariq
    [10] Farman Ali
    [11] Fiza Murtaza
    [12] tayyaba gull tareen
    [13] Muhammad Abubakar
    [14] Fahd Sikandar Khan
    [15] Usman Masud
    [16] Muhammad Ehatisham ul Haq
    [17] Waqas Amin
    [18] SYED TAMOORULHASSAN
    [19] Manzoor Ellahi
    [20] Sarosh Tahir
    [21] M. Usman Abbasi
    [22] Muzzamil Ghaffar
    [23] Qaisar Hussain Alvi
    [24] Suhail Asghar Qureshi
    [25] Muhammad Amjad
    [26] Qamar Abbas
    [27] Shahzad
    [28] Matiullah Ahsan
    [29] Samana Batool
    [30] Asim Masood
    [31] Syed Fasih Ali Gardazi
    [32] Sania Gul
    [33] Muhammad Musad


## **LLM/NLP Pipeline Design**

This section shows the **planned LLM pipeline** (to be fully implemented in Milestone 2).
For now, it demonstrates the prompt structure and simulated output format.

```
Extracted Candidate Data (dict)
    |
    v
[Prompt Builder]      -- Converts dict to natural language prompt
    |
    v
[LLM API Call]        -- GPT-4 / LLaMA / Gemini (Milestone 2)
    |
    v
[Response Parser]     -- Extracts scores and recommendation from JSON response
    |
    v
[Ranking Module]      -- Sorts all candidates by total score
    |
    v
[Final Output]        -- Ranked list + hire/not-hire recommendations
```


In [6]:
# LLM Pipeline Design (Milestone 1 - Structure Demonstration)
# NOTE: Actual API calls will be added in Milestone 2.

def build_llm_prompt(candidate):
    """
    Builds a structured prompt to send to an LLM.
    In Milestone 2, this prompt will be sent to GPT or LLaMA API.
    """
    prompt = (
        'You are an expert HR analyst for an academic/research institution.\n'
        'Evaluate the following candidate and return a JSON response.\n\n'
        f'Name            : {candidate.get("Name", "N/A")}\n'
        f'Applying For    : {candidate.get("Applying For", "N/A")}\n'
        f'Current Job     : {candidate.get("Present Employment", "N/A")}\n'
        f'Current Salary  : {candidate.get("Current Salary", "N/A")}\n'
        f'Expected Salary : {candidate.get("Expected Salary", "N/A")}\n\n'
        'Score the candidate from 0-10 for each:\n'
        '1. Education quality\n'
        '2. Research output (publications, patents)\n'
        '3. Work experience relevance\n'
        '4. Overall suitability\n\n'
        'Respond ONLY in JSON:\n'
        '{\n'
        '  "education_score": 0-10,\n'
        '  "research_score":  0-10,\n'
        '  "experience_score": 0-10,\n'
        '  "overall_score": 0-10,\n'
        '  "recommendation": "Hire" or "Not Hire" or "Shortlist",\n'
        '  "remarks": "brief explanation"\n'
        '}'
    )
    return prompt


def simulate_llm_response(candidate):
    """
    Simulates what the LLM would return.
    In Milestone 2, replace this body with:
        import openai
        response = openai.ChatCompletion.create(
            model='gpt-4',
            messages=[{'role':'user','content': build_llm_prompt(candidate)}]
        )
        return json.loads(response.choices[0].message.content)
    """
    return {
        'candidate':       candidate.get('Name', 'Unknown'),
        'education_score': 7,
        'research_score':  6,
        'experience_score':8,
        'overall_score':   7,
        'recommendation':  'Shortlist',
        'remarks':         'Decent profile. Needs more research publications.'
    }


# Show sample prompt and response
if extracted_data and extracted_data['personal']:
    demo = extracted_data['personal'][0]
else:
    demo = {'Name': 'Dr. Ahmed Khan', 'Applying For': 'Assistant Professor',
            'Present Employment': 'NUST', 'Current Salary': '150000', 'Expected Salary': '200000'}

print('=' * 55)
print('  SAMPLE LLM PROMPT')
print('=' * 55)
print(build_llm_prompt(demo))
print('\n' + '=' * 55)
print('  SIMULATED LLM RESPONSE')
print('=' * 55)
for k, v in simulate_llm_response(demo).items():
    print(f'  {k:<20}: {v}')


  SAMPLE LLM PROMPT
You are an expert HR analyst for an academic/research institution.
Evaluate the following candidate and return a JSON response.

Name            : MUHAMMAD SALMAN QAMAR
Applying For    : Assistant Professor - Electrical & Computer Engineering - SEECS
Current Job     : Associate Professor at Xi'an International University, China , Shaanxi, China since 01/01/2025
Current Salary  : 200000
Expected Salary : 200000

Score the candidate from 0-10 for each:
1. Education quality
2. Research output (publications, patents)
3. Work experience relevance
4. Overall suitability

Respond ONLY in JSON:
{
  "education_score": 0-10,
  "research_score":  0-10,
  "experience_score": 0-10,
  "overall_score": 0-10,
  "recommendation": "Hire" or "Not Hire" or "Shortlist",
  "remarks": "brief explanation"
}

  SIMULATED LLM RESPONSE
  candidate           : MUHAMMAD SALMAN QAMAR
  education_score     : 7
  research_score      : 6
  experience_score    : 8
  overall_score       : 7
  recomme


## **Storage Design**

TALASH stores data in three formats:

| Format | Purpose | File |
|--------|---------|------|
| Excel  | Full data with 8 sheets | `talash_output.xlsx` |
| JSON   | Structured data for LLM | `talash_output.json` |
| CSV    | Flat personal info table | `talash_personal.csv` |


In [7]:
# Save to Excel, JSON, and CSV

def clean_df(df):
    """Remove control characters that would break Excel."""
    return df.map(lambda x: re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', str(x)) if isinstance(x, str) else x)


def save_to_excel(data, path):
    """Save all sections to an Excel file (one sheet per section)."""
    sheet_map = {
        'personal':'Personal Info', 'education':'Education', 'prof_qual':'Prof Qualifications',
        'experience':'Experience', 'publications':'Publications', 'awards':'Awards & Scholarships',
        'patents':'Patents', 'references':'References'
    }
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        for key, sheet_name in sheet_map.items():
            df = pd.DataFrame(data[key])
            if df.empty: df = pd.DataFrame(columns=['candidate_id', 'No Data'])
            clean_df(df).to_excel(writer, sheet_name=sheet_name, index=False)
    print(f'  Excel saved : {path}')


def save_to_json(data, path):
    """Save all data to JSON file."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print(f'  JSON saved  : {path}')


def save_to_csv(data, path):
    """Save personal info as a flat CSV."""
    df = pd.DataFrame(data['personal'])
    if not df.empty:
        clean_df(df).to_csv(path, index=False, encoding='utf-8-sig')
        print(f'  CSV saved   : {path}')
    else:
        print('  No personal data to save to CSV.')


if extracted_data:
    print('[Saving output files...]')
    save_to_excel(extracted_data, OUTPUT_EXCEL)
    save_to_json(extracted_data, OUTPUT_JSON)
    save_to_csv(extracted_data, OUTPUT_CSV)
    print('\nAll files saved!')
else:
    print('No data to save. Run the pipeline cell first.')


[Saving output files...]
  Excel saved : talash_output.xlsx
  JSON saved  : talash_output.json
  CSV saved   : talash_personal.csv

All files saved!



##  **Display Extracted Results**

View the extracted data as tables right here in the notebook.


In [8]:
# Display Results as Tables

def display_results(data):
    """Display extracted sections as pandas DataFrames."""
    if not data:
        print('No data to display.')
        return
    sections = [
        ('personal',     'Personal Information'),
        ('education',    'Education'),
        ('experience',   'Work Experience'),
        ('publications', 'Research Publications'),
        ('awards',       'Awards and Scholarships'),
        ('patents',      'Patents'),
    ]
    for key, title in sections:
        records = data.get(key, [])
        print(f'\n{"="*55}')
        print(f'  {title}')
        print(f'{"="*55}')
        if records:
            df = pd.DataFrame(records)
            display(df.head(5))
            if len(records) > 5:
                print(f'  ... and {len(records)-5} more rows')
        else:
            print('  (No data extracted for this section)')


if extracted_data:
    display_results(extracted_data)
else:
    print('Run the pipeline cell first to see results here.')



  Personal Information


,candidate_id,Name,Date of Birth,Marital Status,Current Salary,Expected Salary,Present Employment,Applying For,Apply Date,Source File
0,1,MUHAMMAD SALMAN QAMAR,10-Nov-1988 /,Married,200000,200000,Associate Professor at Xi'an International Uni...,Assistant Professor - Electrical & Computer En...,22-Oct-2025,Handler (8).pdf
1,2,Aamina Akbar,18-Nov-1987 / US Father,Married,200000,200000,Unemployed,Assistant Professor - Electrical & Computer En...,25-Oct-2025,Handler (8).pdf
2,3,MUHAMMAD SHAHWAZ,23-Feb-1988 / PAKISTANI Father,Un-Married,00,00,Unemployed,Assistant Professor - Electrical & Computer En...,20-Oct-2025,Handler (8).pdf
3,4,shaheer,18-Feb-1978 / pakistaniFather,Married,350000,350000,"at city University , peshawar since 05/01/2023",Assistant Professor - Electrical & Computer En...,24-Oct-2025,Handler (8).pdf
4,5,Nasir Ali Shah,06-Apr-1991 /,Married,350000,350000,"Postdoctoral researcher at Orange Innovation ,...",Assistant Professor - Electrical & Computer En...,25-Oct-2025,Handler (8).pdf


  ... and 38 more rows

  Education


,candidate_id,Degree/Details,Year,Grade/CGPA
0,1,PhD in Electrical EngineeringCommunication Sys...,2024,33
1,1,MS in Electrical EngineeringCommunication Syst...,2014,03
2,1,BSc Electronics EngineeringElectronics 69.84 2...,2011,69.84
3,1,HSSC Pre-Engineering 65.45 2006 B.I.S.E Bannu,2006,65.45
4,1,SSc Science Subjects 71 2004 B.I.S.E Bannu,2004,71


  ... and 171 more rows

  Work Experience


...

,candidate_id,Title,Published In,Impact Factor,Date
0,1,A Hybrid Framework Integrating Deterministic C...,International Journal3 2.10 84 5463-,2.10,Jul-2025
1,1,AI for Cleaner Air: PredictiveModeling of PM2....,International Journal3 2.50 144 3557-,2.50,Sep-2025
2,1,References Improvement of Traveling Salesman P...,International Journal11 2.80 11 4780May-,2.80,2021
3,1,Enhancing Energy Efficiency in WSNs with Hybri...,international Conference0.00 0 1-6 Jul-,N/A,2024
4,1,A Novel Approach to Energy Optimization: Effic...,International Journal2 2.10 79 2945-,2.10,May-2024


  ... and 426 more rows

  Awards and Scholarships


,candidate_id,Type,Detail
0,2,Other,References NOMA and 5G emerging technologies: ...
1,2,Merit Scholarships,Awarded HEC Faculty development scholarship pr...
2,2,Merit Scholarships,Merit scholarship by Army Public School and Co...
3,2,Membership,Pakistan Engineering Council Number-ELECT/32735
4,2,Merit Scholarships,Merit scholarship in MS Electrical(Telecom) En...


  ... and 169 more rows

  Patents


,candidate_id,Patent Details
0,6,References Spectral processing of self- mixing...
1,9,References No Patent Title 0 NA Name Contact T...
2,20,Exploiting Transmission Level Potentialities i...
3,42,A Temporal Broadening- Aware Pulse Width Adapt...
